# RRUFF Raman classification Project

## Getting the dataset ready

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
%matplotlib inline

## Opening the file
to explore

In [ ]:
with open("C:/Users/68pup/pythstuff/mycodes/ML/rruff_new/Actinolite__R040063-4__Raman__514__0-000____Raman_Data_Processed__c801aaca84d830ae65447fa8483e.txt","r") as f:
    for line in f:
        print(line.strip())

In [ ]:
with open("C:/Users/68pup/pythstuff/mycodes/ML/rruff_new/Actinolite__R040063-4__Raman__514__0-000____Raman_Data_Processed__c801aaca84d830ae65447fa8483e.txt","r") as f:
    for line in f:
        if line.startswith("##") or line == "":
            continue
        print(line.strip())

### Defined a function to extract info from the file
the wavenumbers and intensities. It ignores the metadata starting with "##" and also any empty line. Also strips the data. Then in each iteration(where it is looking at one line at a time), it makes a python list "parts" storing in the values separated by the given delimiter, ie "," here. 
then we had defined empty lists for wavenumber and intensity info, so it in the loop when the parts lists has only two items in it for each line, the first part is the wavenumber that gets appended to the respective list with conversion to a float value, similarly the second item in the parts list is the intensity value which gets floated and appended in the respective list.


In [ ]:
def parse_rruff_file(filepath):
    wavenumbers = []
    intensity = []
    with open(filepath,"r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("##") or line == "" :
                continue
            parts = line.split(",")
            wavenumbers.append(float(parts[0]))
            intensity.append(float(parts[1]))
    return np.array(wavenumbers),np.array(intensity)
waven,intenn = parse_rruff_file("C:/Users/68pup/pythstuff/mycodes/ML/rruff_new/Actinolite__R040063-4__Raman__514__0-000____Raman_Data_Processed__c801aaca84d830ae65447fa8483e.txt")
print(waven,intenn)

### getting the name for the particular mineral from the file name
os.path.basename strips the directory, .split("_")[0] takes the first chunk before the underscore delimiter.

In [ ]:
filepath  = "C:/Users/68pup/pythstuff/mycodes/ML/rruff_new/Actinolite__R040063-4__Raman__514__0-000____Raman_Data_Processed__c801aaca84d830ae65447fa8483e.txt"
filename = os.path.basename(filepath)
min_name  = filename.split("_")[0]
print(min_name)

In [ ]:
comn_waven = np.arange(100,1550,2)

interp1d builds a function from your (x, y) data points. You then call that function at any x value you want — here, at every point in common_wavenumbers. bounds_error=False, fill_value=0 means if the common axis goes slightly outside the spectrum's range, it fills with 0 instead of crashing.


In [ ]:
def interpolate_spectrum(wn,inten,cwn):
    f = interp1d(wn,inten,kind="linear",bounds_error=False,fill_value=0)
    return f(cwn)

interpolate_spectrum(waven, intenn, comn_waven)

## Checking if the interpolation worked or not
### because the different spectras will be all at different wavenumber values so we want a common axis

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(waven,intenn)
plt.plot(comn_waven,interpolate_spectrum(waven, intenn, comn_waven))
plt.show()

## the full dataset builder
### now we define a function to take in all the txt files and then do the similar parsing as above and build a ML ready dataset 

glob.glob finds all files matching a pattern. The loop builds two lists in parallel — rows (spectral data) and labels (mineral names). At the end, pd.DataFrame(rows, columns=common_wn) turns the list of arrays into a table where each column is a wavenumber.

In [ ]:
def datasetbuilder(folderpath, minerals_to_include, common_wn):
    rows = []
    labels = []
    # taking in all the txt files form the folder into one variable
    allfiles = glob.glob(os.path.join(folderpath,"*.txt"))
    for filepath in allfiles:
        mineral = os.path.basename(filepath).split("_")[0]
        if mineral not in minerals_to_include:
            continue
        waven,intenn = parse_rruff_file(filepath)
        resampled = interpolate_spectrum(waven,intenn,common_wn)
        rows.append(resampled)
        labels.append(mineral)
    df = pd.DataFrame(rows,columns=common_wn)
    df["label"] = labels
    return df

def parse_rruff_file(flpath):
    w = []
    i = []
    with open(flpath,"r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("##") or line=="":
                continue
            parts = line.split(",")
            if len(parts)>2:
                continue
            try:
                w.append(float(parts[0]))
                i.append(float(parts[1]))
            except ValueError:
                continue
    return np.array(w), np.array(i)
    
def interpolate_spectrum(wn,it,cwn):
    f = interp1d(wn,it,kind="linear",bounds_error=False,fill_value=0)
    return f(cwn)

minerals = ["Actinolite","Adamite","Aegirine","Aenigmatite","Akermanite","Albite"]
folderpath="C:/Users/68pup/pythstuff/mycodes/ML/rruff_new"
comnwn = np.arange(70,1700,2)
df1 = datasetbuilder(folderpath,minerals,comnwn)
    

In [ ]:
df1

## Now that the data is ready, we do somethings before getting to the Modeling

# Exploratory Data Analysis